# Sesión 4 — Regresión lineal: qué hay dentro del `.fit()`

**Introducción al Análisis Avanzado de Datos con Python** · Univalle · Módulo 2 — Modelos supervisados

## Objetivos de hoy
Al terminar la sesión usted podrá:
1. Ajustar una regresión lineal, **leer sus coeficientes** y explicar una predicción como una suma.
2. Explicar qué minimiza el `.fit()` (la suma de residuales al cuadrado) y cómo lo encuentra: **fórmula cerrada** o **descenso de gradiente** con tasa de aprendizaje.
3. Evaluar un modelo de regresión con **MAE, RMSE y R²** contra una línea base.
4. Usar **Ridge y Lasso** para controlar coeficientes y elegir variables, y reconocer cuándo un R² es demasiado bueno para ser cierto.

> 💾 **Antes de escribir nada:** menú *Archivo → Guardar una copia en Drive* y trabaje en esa copia; si no, lo que escriba se pierde al cerrar la pestaña. Si Colab avisa que el notebook no es de Google, elija *Ejecutar de todos modos*.

## Dónde estamos en el pipeline

```
 ┌────────┐   ┌──────────────────┐   ┌────────┐   ┌────────────┐   ┌────────────────┐
 │ DATOS  │ → │ PREPROCESAMIENTO │ → │ MODELO │ → │ EVALUACIÓN │ → │ INTERPRETACIÓN │
 └────────┘   └──────────────────┘   └────────┘   └────────────┘   └────────────────┘
                                       ▲▲▲▲▲▲▲▲
   HOY: un modelo nuevo y, por primera vez, abrimos su .fit() para ver la optimización por dentro.
```

## Cómo usar este notebook
- 🔍 **Predice** antes de ejecutar · 👀 **Mire** (piénselo un momento antes de seguir) · ✏️ **Completa**: desde hoy el andamiaje es **medio** (usted escribe bloques cortos con la firma dada) · 💥 **Trampa** · 🖼️ **Fichas** · ⚡ **Mini-reto**.

> **Dos datasets hoy.** *Insurance* (1 338 asegurados de EE. UU., cargo médico anual; el clásico de Lantz y de PyCaret) para ver el método con claridad. Luego **Saber 11, Valle del Cauca, 2022** (78 720 estudiantes; ICFES vía datos.gov.co) para aplicarlo con datos colombianos de verdad.


## 0. Setup

Baja del repositorio del curso los dos datasets y `figuras.py`. Si ya están en el disco de Colab, no los vuelve a bajar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
REPO = "https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/"
ARCHIVOS = {"insurance.csv": "datos/insurance.csv",
            "saber11_valle_2022.csv.gz": "datos/saber11_valle_2022.csv.gz",
            "figuras.py": "tools/figuras.py"}

def asegurar_datos(archivo, url_repo):
    """Deja un archivo en el disco de Colab: lo baja del repositorio del curso si aún no está."""
    from pathlib import Path
    import requests
    if Path(archivo).exists():
        return True
    try:
        r = requests.get(url_repo, timeout=120)
        if r.ok:
            Path(archivo).write_bytes(r.content)
            return True
    except Exception:
        pass
    print(f"⚠️ No pude bajar {archivo}. Use el Plan B (celda siguiente).")
    return False

for archivo, ruta in ARCHIVOS.items():
    asegurar_datos(archivo, REPO + ruta)
try:
    import figuras as fg
except ModuleNotFoundError:   # Plan B: GitHub no respondió y no llegó figuras.py
    from google.colab import files
    print("⚠️ No llegó figuras.py: súbalo (el profesor lo comparte por el canal del curso).")
    files.upload()
    import figuras as fg
print("Listo:", ", ".join(ARCHIVOS))

<details><summary><b>Plan B</b> — si la celda anterior avisó que no pudo bajar algún archivo</summary>

La celda siguiente revisa sola qué falta y solo en ese caso le pide subirlo (el instructor comparte los archivos por el grupo del curso).
</details>

In [ ]:
# Plan B: solo actúa si falta algún archivo. Con "Ejecutar todo" no se detiene.
from pathlib import Path
faltan = [a for a in ARCHIVOS if not Path(a).exists()]
if not faltan:
    print("✓ Todo disponible: no hace falta el Plan B.")
else:
    from google.colab import files
    print("⚠️ Faltan:", faltan, "→ súbalos")
    files.upload()

In [ ]:
fg.pipeline("modelo", modulo=2, subtitulo="HOY: un modelo nuevo (regresión lineal) y qué pasa dentro de su .fit()");

## 1. ¿Cuánto paga en salud un asegurado? — Insurance

| Columna | Qué es |
|---|---|
| `age`, `bmi`, `children` | edad, índice de masa corporal, número de hijos |
| `sex`, `smoker`, `region` | sexo, si fuma, región de EE. UU. |
| `charges` | **la y**: cargo médico anual en USD |

In [ ]:
ins = pd.read_csv("insurance.csv")
print(ins.shape, "| filas duplicadas:", ins.duplicated().sum())
ins = ins.drop_duplicates()          # la Sesión 2 en una línea: una fila repetida
ins.head()

👀 **Mire:** ¿cuánto paga **en promedio** al año alguien que fuma, comparado con alguien que no fuma? ¿El doble? ¿El triple? ¿Más?

In [ ]:
ins.groupby("smoker")["charges"].agg(["mean", "median", "count"]).round(0)

## 2. Antes del código: ¿qué recta es "la mejor"?

Empecemos con **una sola variable**: la edad, solo para quienes no fuman. Una regresión lineal propone

$$\hat{y} = \beta_0 + \beta_1 \cdot \text{edad}$$

¿Cuál de todas las rectas posibles? La que comete **menos error**. Cada punto tiene un **residual** $y_i - \hat{y}_i$ (la línea roja vertical) y la regresión minimiza la suma de sus cuadrados.

In [ ]:
no_fuman = ins[ins["smoker"] == "no"]
muestra = no_fuman.sample(40, random_state=3)
fg.recta_residuales(muestra["age"], muestra["charges"] / 1000, malas=((no_fuman["charges"].mean() / 1000, 0.0), None),
                    xlabel="edad", ylabel="cargo anual (miles de USD)");

### 📐 Fundamento

$$\text{pérdida}(\beta_0, \beta_1) = \frac{1}{n}\sum_{i=1}^{n}\big(y_i - (\beta_0 + \beta_1 x_i)\big)^2 \qquad \text{(error cuadrático medio, MSE)}$$

- $y_i$: el cargo real de la persona $i$ · $\beta_0 + \beta_1 x_i$: lo que la recta predice para ella.
- ¿Por qué **al cuadrado** y no el valor absoluto? Castiga más los errores grandes y, sobre todo, deja una pérdida con forma de **tazón liso**: tiene un solo fondo y se puede encontrar con álgebra o rodando cuesta abajo.

<details><summary><b>Para quien quiera más:</b> la fórmula cerrada</summary>

Con varias variables, en forma matricial, el fondo del tazón se despeja en un paso:
$$\hat{\beta} = (X^\top X)^{-1} X^\top y$$
Es lo que hace `LinearRegression` por dentro. Necesita invertir una matriz de $p \times p$ (con $p$ variables): con millones de filas o miles de variables se vuelve costoso, y ahí entra el descenso de gradiente.
</details>

![Dentro del .fit() — regresión lineal](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S04/img/dentro_del_fit_lineal.png)

## 3. Cómo se encuentra el fondo: descenso de gradiente

Cada par $(\beta_0, \beta_1)$ es una recta posible, y cada recta tiene su error. Si dibujamos el error de **todas** las rectas, sale un paisaje con forma de tazón. Entrenar es encontrar el fondo.

(Para que el tazón se vea redondo usamos la edad **escalada** —restar la media, dividir por la desviación—, como en la Sesión 2, y el cargo en miles de USD.)

> 🧪 **Laboratorio** (para jugar después de clase, o ya si va adelantado): [**Descenso de gradiente**](https://nitobest.github.io/curso-aad-python/laboratorio/descenso.html) · la bolita rueda por el tazón del error; mueva la tasa de aprendizaje y vea cuándo se sale. Tres pestañas: **Mira**, **Juega** y **Reto** (prediga antes de correr). Funciona en el celular.

In [ ]:
x = ((no_fuman["age"] - no_fuman["age"].mean()) / no_fuman["age"].std()).values   # edad escalada
y_miles = (no_fuman["charges"] / 1000).values                                          # cargo en miles
fg.superficie_perdida(x, y_miles);

### La idea: rodar cuesta abajo
Desde cualquier punto del tazón, el **gradiente** $\nabla L$ dice hacia dónde sube la pérdida más rápido. Damos un pasito en la dirección contraria, y repetimos:

$$\beta \leftarrow \beta - \eta \, \nabla L(\beta)$$

Léala así: *"muévase un poquito ($\eta$) en contra de la pendiente"*. Para nuestra recta, las dos derivadas son:

$$\frac{\partial L}{\partial \beta_0} = \frac{2}{n}\sum(\hat{y}_i - y_i) \qquad \frac{\partial L}{\partial \beta_1} = \frac{2}{n}\sum(\hat{y}_i - y_i)\,x_i$$

### ✏️ Completa — escriba usted un paso del descenso
`residuo` es $\hat{y} - y$. Complete las dos líneas del `return` con las fórmulas de arriba.

In [ ]:
def paso_gradiente(b0, b1, x, y, eta):
    """Un paso de descenso de gradiente para la recta y ≈ b0 + b1·x (pérdida: MSE)."""
    residuo = (b0 + b1 * x) - y
    grad_b0 = ...   # TODO: 2 × promedio del residuo
    grad_b1 = ...   # TODO: 2 × promedio de (residuo × x)
    return b0 - eta * grad_b0, b1 - eta * grad_b1

b0, b1 = 0.0, 0.0                       # arrancamos en una recta plana en cero
for k in range(100):
    b0, b1 = paso_gradiente(b0, b1, x, y_miles, eta=0.1)
    if k in (0, 1, 4, 19, 99):
        print(f"paso {k+1:>3}: b0 = {b0:6.3f}   b1 = {b1:6.3f}   MSE = {np.mean((y_miles - (b0 + b1 * x))**2):7.2f}")

Ahora lo **vemos**: arriba, la recta en distintos pasos sobre los datos; abajo, el camino sobre el tazón y la pérdida que baja.

In [ ]:
fg.descenso_gradiente_paso_a_paso(x, y_miles, eta=0.1, mostrar=(0, 1, 3, 10, 30, 60));

### ¿Llegamos al mismo sitio que la fórmula cerrada?

In [ ]:
from sklearn.linear_model import LinearRegression

cerrada = LinearRegression().fit(x.reshape(-1, 1), y_miles)
print(f"fórmula cerrada (LinearRegression): b0 = {cerrada.intercept_:.3f}   b1 = {cerrada.coef_[0]:.3f}")
print(f"descenso de gradiente, 100 pasos:   b0 = {b0:.3f}   b1 = {b1:.3f}")

### 🔍 Predice — la perilla del optimizador
El paso de cada iteración es $\eta$ (la **tasa de aprendizaje**). Arriba usamos 0,1 con la edad bien escalada y llegó al fondo. Ahora cambiamos dos cosas: la edad va en **unidades de 30 años** (el tazón se **alarga**) y solo damos **40 pasos**. ¿Qué cree que pasa con cada $\eta$? (no llega · llega · zigzaguea · diverge)

**Su predicción:** η = 0,1: ____ · η = 0,5: ____ · η = 0,95: ____ · η = 1,02: ____

In [ ]:
x30 = ((no_fuman["age"] - no_fuman["age"].mean()) / 30).values   # edad en unidades de 30 años: el tazón se alarga
fg.learning_rate_efecto(x30, y_miles, etas=(0.1, 0.5, 0.95, 1.02), pasos=40);

La tasa de aprendizaje es una perilla **del optimizador**, no del modelo: no cambia qué recta es la mejor, solo cómo se llega a ella. Y el descenso de gradiente es una forma de optimizar entre varias:

![Optimización: el mapa](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S04/img/mapa_optimizacion.png)

## 4. El modelo completo: todas las variables de Insurance

Con varias variables la idea es la misma, en más dimensiones: $\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots$

**One-hot con `drop_first=True`.** En la Sesión 2 creamos una columna por categoría. En un modelo lineal **con intercepto**, una de ellas sobra: si sé que alguien no es del noroeste, ni del sureste, ni del suroeste, ya sé que es del noreste. Esa categoría omitida queda como **referencia** y cada coeficiente se lee *"comparado con la referencia"*.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor

X_ins = pd.get_dummies(ins.drop(columns="charges"), drop_first=True, dtype=int)
y_ins = ins["charges"]
X_tr, X_te, y_tr, y_te = train_test_split(X_ins, y_ins, test_size=0.2, random_state=42)
list(X_ins.columns)

### ✏️ Completa — entrene el modelo y su línea base
La línea base de regresión es `DummyRegressor()`: predice siempre **el promedio** de entrenamiento.

In [ ]:
modelo = ...   # TODO: LinearRegression entrenado con X_tr, y_tr
base = ...     # TODO: DummyRegressor entrenado con X_tr, y_tr
print("Modelo y línea base entrenados.")

### ¿Qué tan bien predice? Tres métricas

![¿Qué tan bien predice un modelo de regresión?](https://raw.githubusercontent.com/Nitobest/curso-aad-python/main/sesiones/S04/img/metricas_regresion.png)

### ✏️ Completa — una función que evalúa cualquier modelo de regresión
La va a usar el resto de la sesión. Recibe un nombre, un modelo **ya entrenado** y unos datos; predice y escribe en **una línea** las tres métricas de la ficha (`mean_absolute_error`, `mean_squared_error` —sáquele raíz para el RMSE— y `r2_score`), con un decimal en MAE y RMSE y tres en R².

<details><summary>Pista</summary>

```python
p = m.predict(X)
print(f"{nombre:<22} MAE = {mean_absolute_error(y, p):>8,.1f}   RMSE = {mean_squared_error(y, p) ** 0.5:>8,.1f}   R² = {r2_score(y, p):6.3f}")
```
</details>

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluar(nombre, m, X, y):
    """Imprime en una línea MAE, RMSE y R² del modelo m (ya entrenado) sobre los datos X, y."""
    ...   # TODO: prediga con m y escriba las tres métricas en una línea

evaluar("línea base (promedio)", base, X_te, y_te)
evaluar("regresión lineal", modelo, X_te, y_te)

### Los coeficientes
👀 **Mire:** antes de verlos, ¿cuánto **suma** al cargo anual ser fumador, según el modelo (en USD)? Recuerde la tabla de la sección 1.

In [ ]:
coefs = pd.Series(modelo.coef_, index=X_ins.columns).round(0)
print(f"intercepto: {modelo.intercept_:,.0f}")
coefs.sort_values()

### Lo que aprendió, dibujado sobre los datos

In [ ]:
fg.recta_por_grupo(ins.assign(smoker_yes=(ins["smoker"] == "yes").astype(int)), "age", "charges", "smoker",
                   modelo, list(X_ins.columns), etiquetas={"no": "no fuma", "yes": "fuma"},
                   xlabel="edad (años)", ylabel="cargo médico anual (USD)", titulo_grupo="¿fuma?");

### Una fila entra, una predicción sale
Un modelo lineal se explica **sumando**: intercepto + coeficiente × valor de cada variable.

In [ ]:
i = 0
fg.contribuciones_una_fila(modelo, X_te.iloc[i], X_ins.columns, real=y_te.iloc[i], unidad=" USD");

### ⚡ Mini-reto (opcional)
En la figura de las rectas, los fumadores forman **dos nubes** y la recta roja pasa entre ellas, sin acertarle a ninguna. ¿Qué variable las separa? Pista: grafique `charges` contra `age` solo para fumadores, coloreando por `bmi > 30`. ¿Qué le diría eso sobre la suposición de "rectas paralelas"? (La respuesta tiene nombre: **interacción**; la retomamos con los árboles en la Sesión 6.)

## 5. Ahora con datos colombianos: Saber 11, Valle del Cauca, 2022

78 720 estudiantes que presentaron la prueba en el segundo semestre de 2022 (calendario A) en colegios del Valle. La pregunta: **¿cuánto del puntaje global se explica por las condiciones del estudiante, su familia y su colegio?**

In [ ]:
s11 = pd.read_csv("saber11_valle_2022.csv.gz")
print(s11.shape)
s11.head(3)

### ✏️ Completa — aplique la Sesión 2
Mire rápido el dataset: ¿qué columnas tienen vacíos y cuántos? (una línea). Fíjese en `fami_estratovivienda` y en las de educación de los padres.

In [ ]:
...   # TODO: % de vacíos por columna, de mayor a menor (Sesión 2)

In [ ]:
print(s11["fami_estratovivienda"].value_counts(dropna=False))
print("\npunt_global:", s11["punt_global"].describe().round(1).to_dict())

Para X usamos variables del colegio y de la familia, y la edad. Las categóricas van a *one-hot* con `dummy_na=True` (el vacío es una categoría más, como en la Sesión 2) y `drop_first=True` (sección 4): en cada variable la **primera categoría en orden alfabético** queda como **referencia**; en el estrato es *Estrato 1*. Dos detalles más: las columnas que valen lo mismo en todas las filas (el "vacío" de variables que no tienen vacíos) no aportan nada y se quitan, y la edad faltante (32 estudiantes) se llena con la mediana **de entrenamiento** (Sesión 3). La *y* es `punt_global` (0 a 500).

In [ ]:
categoricas = ["cole_area_ubicacion", "cole_bilingue", "cole_caracter", "cole_jornada", "cole_naturaleza", "estu_genero",
               "fami_educacionmadre", "fami_educacionpadre", "fami_estratovivienda", "fami_tieneautomovil",
               "fami_tienecomputador", "fami_tieneinternet", "fami_tienelavadora", "fami_personashogar", "fami_cuartoshogar"]
X_s = pd.get_dummies(s11[categoricas + ["edad"]], columns=categoricas, dummy_na=True, drop_first=True, dtype=int)
X_s = X_s.loc[:, X_s.nunique() > 1]                    # fuera las columnas constantes
y_s = s11["punt_global"]
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(X_s, y_s, test_size=0.2, random_state=42)
mediana_edad = Xs_tr["edad"].median()                  # la mediana sale SOLO de entrenamiento
Xs_tr, Xs_te = Xs_tr.fillna({"edad": mediana_edad}), Xs_te.fillna({"edad": mediana_edad})
print("X:", X_s.shape)

👀 **Mire:** con 65 columnas socioeconómicas y del colegio, ¿qué R² cree que logra la regresión lineal? ¿0,2? ¿0,5? ¿0,8?

In [ ]:
lin_s = LinearRegression().fit(Xs_tr, ys_tr)
base_s = DummyRegressor().fit(Xs_tr, ys_tr)
evaluar("línea base (promedio)", base_s, Xs_te, ys_te)
evaluar("regresión lineal", lin_s, Xs_te, ys_te)

### 🔍 Predice — ¿le cree al coeficiente?
Comparado con un estudiante de **estrato 1** (la categoría de referencia), y con todo lo demás igual, ¿cuántos puntos **más o menos** cree que predice el modelo para uno de **estrato 6**?

**Su predicción:** ____

In [ ]:
estrato = pd.Series(lin_s.coef_, index=X_s.columns).filter(like="estrato").round(1)
print("Puntos comparado con Estrato 1 (la referencia):")
print(estrato.sort_values())
print("\nPromedio real de punt_global por estrato:")
print(s11.groupby("fami_estratovivienda")["punt_global"].agg(["mean", "count"]).round(1))

**¡Estrato 6 unos 30 puntos por debajo de estrato 1!** Y no es un error del modelo: el promedio crudo también lo dice (213 contra 243). Antes de publicar *"en el Valle, los estudiantes de estrato 6 sacan menos"*, la pregunta de la Sesión 2: **¿cambió el mundo o cambió el dato?**

In [ ]:
perfil = s11[s11["fami_estratovivienda"].isin(["Estrato 1", "Estrato 3", "Estrato 6"])].groupby("fami_estratovivienda").agg(
    estudiantes=("punt_global", "size"),
    colegio_oficial=("cole_naturaleza", lambda v: (v == "OFICIAL").mean()),
    tiene_internet=("fami_tieneinternet", lambda v: (v == "Si").mean()),
    tiene_carro=("fami_tieneautomovil", lambda v: (v == "Si").mean()),
).round(2)
perfil

## 6. Regularización: Ridge y Lasso

Con 65 columnas, muchas parecidas entre sí (educación de la madre y del padre, computador e internet), los coeficientes se vuelven **inestables** y difíciles de leer. La regularización agrega a la pérdida un **castigo por coeficientes grandes**:

$$\text{Ridge: } \sum (y_i - \hat{y}_i)^2 + \alpha \sum_j \beta_j^2 \qquad\qquad \text{Lasso: } \frac{1}{2n}\sum (y_i - \hat{y}_i)^2 + \alpha \sum_j |\beta_j|$$

(Así las define scikit-learn: Lasso divide la suma de cuadrados por $2n$ y Ridge no. **Por eso el $\alpha$ de Lasso y el de Ridge no están en la misma escala**: con nuestros 62 976 estudiantes de entrenamiento, un $\alpha$ de Lasso equivale a uno unas 126 000 veces más grande en Ridge. La figura de abajo los pone en el mismo eje.)

- $\alpha$ es la **fuerza** del castigo: $\alpha = 0$ es la regresión de siempre; $\alpha$ muy grande aplasta todos los coeficientes a 0.
- **Ridge** encoge todos los coeficientes; **Lasso** apaga los menos útiles (los vuelve exactamente 0): **elige variables**.
- Ojo: el castigo depende del tamaño del coeficiente, y el tamaño depende de la escala de la variable. Por eso **se escala antes** (Sesión 2). Aquí $\alpha$ es una perilla **del modelo**, no del optimizador.

<details><summary><b>Para quien quiera más:</b> ¿por qué Lasso apaga y Ridge no?</summary>

El castigo $|\beta|$ tiene una "punta" en 0: cerca de cero, sigue empujando con la misma fuerza hasta dejar el coeficiente exactamente en 0. El castigo $\beta^2$ se vuelve muy suave cerca de 0 (su pendiente es $2\beta$, que se anula), así que encoge pero casi nunca llega a cero.
</details>

### 🔍 Predice
Si subimos $\alpha$ poco a poco, ¿qué pasará con el R² de prueba? ¿Baja de inmediato, se mantiene un rato, sube?

**Su predicción:** ____

In [ ]:
fg.regularizacion_alpha(Xs_tr, ys_tr, Xs_te, ys_te, alphas=np.logspace(-2, 1.2, 12));

**Lo que pasa:** el R² **se mantiene** un buen tramo mientras los coeficientes se encogen o se apagan, y solo cae cuando el castigo ya es muy fuerte. Muchas variables sobraban.

### ✏️ Completa — Lasso como selector de variables
Entrene un `Lasso` con `alpha=2` (y `max_iter=10000`, para que el optimizador alcance a llegar) sobre las variables **escaladas** de entrenamiento (el escalador ya se ajustó solo con entrenamiento, Sesión 3). Luego arme `vivos`: la Serie de coeficientes con los nombres de las columnas, quedándose solo con los distintos de cero.

<details><summary>Pista</summary>

```python
lasso = Lasso(alpha=2, max_iter=10000).fit(Zs_tr, ys_tr)
vivos = pd.Series(lasso.coef_, index=X_s.columns)
vivos = vivos[vivos != 0]
```
</details>

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

escalador = StandardScaler().fit(Xs_tr)
Zs_tr, Zs_te = escalador.transform(Xs_tr), escalador.transform(Xs_te)
lasso = ...   # TODO: Lasso con alpha=2 entrenado sobre las variables escaladas de entrenamiento
vivos = ...   # TODO: coeficientes de lasso con su nombre de columna, solo los distintos de 0
print(f"Lasso conserva {len(vivos)} de {X_s.shape[1]} variables · R² prueba = {lasso.score(Zs_te, ys_te):.3f} (lineal: {lin_s.score(Xs_te, ys_te):.3f})")
vivos.reindex(vivos.abs().sort_values(ascending=False).index).head(8).round(2)

## 7. 💥 Cinco columnas sin usar

El R² de 0,18 deja un sabor amargo. En el dataset hay cinco columnas más que no hemos usado: los **puntajes por área** (matemáticas, lectura crítica, sociales, ciencias naturales, inglés). Agreguémoslas a X a ver cuánto mejora.

### 🔍 Predice
¿Qué R² cree que logra el modelo con los puntajes por área?

**Su predicción:** R² ≈ ____

In [ ]:
areas = ["punt_matematicas", "punt_lectura_critica", "punt_sociales_ciudadanas", "punt_c_naturales", "punt_ingles"]
X_mas = pd.concat([X_s, s11[areas]], axis=1)
Xm_tr, Xm_te, _, _ = train_test_split(X_mas, y_s, test_size=0.2, random_state=42)
medianas = Xm_tr.median()                              # otra vez: medianas solo de entrenamiento
Xm_tr, Xm_te = Xm_tr.fillna(medianas), Xm_te.fillna(medianas)
lin_mas = LinearRegression().fit(Xm_tr, ys_tr)
evaluar("lineal + puntajes", lin_mas, Xm_te, ys_te)

In [ ]:
fg.pred_vs_real({"socioeconómicas (honesto)": (ys_te, lin_s.predict(Xs_te)),
                 "+ puntajes por área": (ys_te, lin_mas.predict(Xm_te))}, unidad=" pts");

**R² = 1,000 y un error de 0,2 puntos.** El modelo perfecto. ¿Qué aprendió? Miremos los coeficientes de los puntajes:

In [ ]:
pd.Series(lin_mas.coef_, index=X_mas.columns)[areas].round(3)

## 8. Cierre — qué cambió hoy en el pipeline

| Etapa | Hoy (Sesión 4) |
|---|---|
| Datos | dos datasets: uno de referencia (Insurance) y uno colombiano (Saber 11); duplicados y vacíos con lo de la Sesión 2 |
| Preprocesamiento | `drop_first` en modelos con intercepto · escalar antes de Ridge/Lasso y del descenso de gradiente |
| Modelo | **regresión lineal**: pérdida cuadrática · fórmula cerrada vs **descenso de gradiente** · **tasa de aprendizaje** · Ridge y Lasso |
| Evaluación | **MAE, RMSE, R²** contra `DummyRegressor` · un R² ≈ 1 es sospechoso |
| Interpretación | coeficientes "manteniendo lo demás fijo" · una predicción = suma de contribuciones · un coeficiente raro es primero una pregunta sobre el dato |

**Dentro del `.fit()` — lo que ya puede comparar:**

| | Árbol (S1–S3) | Regresión lineal (S4) |
|---|---|---|
| Qué minimiza | Gini de cada corte | suma de residuales² (+ castigo en Ridge/Lasso) |
| Cómo lo busca | búsqueda voraz (exhaustiva en cada corte) | fórmula cerrada o descenso de gradiente |
| Qué perilla controla la búsqueda | ninguna | tasa de aprendizaje η (si hay gradiente) |
| Perilla del modelo | `max_depth` | `alpha` |

**Para la Sesión 5 — clasificación:** volvemos a una *y* de sí/no, con un dataset nuevo (una campaña de mercadeo bancario). ¿Qué pasa si a una pregunta de sí/no le ajustamos una recta? Spoiler: se sale de 0 y 1, y hay que doblarla.